In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from ipywidgets import interact
from pyspark.sql import functions as F

In [0]:
sql_query = """SELECT cy.Year, cy.Country, cc.ISO3, cy.Item, cy.ItemCode, cy.Element, cy.Unit, cy.Yield, avg_temp.Temperature, corr_index.CPI as CorruptionIndex, sog.GDP, up.UrbanPopulation, up.RuralPopulation, fu.FertilizerUse,
peia.PeopleEmployedInAgriculture, fw.FreshwaterWithdrawal
FROM agriculture_db.crop_yield cy
INNER JOIN agriculture_db.country_codes cc ON cc.ISONum = cy.AreaCodeM49
INNER JOIN 
(SELECT AVG(ast.Temperature) as Temperature, ast.Year, ast.ISO3
FROM agriculture_db.average_surface_temperature ast
GROUP BY ast.Country, ast.Year, ast.ISO3) avg_temp ON avg_temp.ISO3 = cc.ISO3 AND avg_temp.Year = cy.Year
INNER JOIN agriculture_db.corruption_index corr_index ON corr_index.ISO = cc.ISO3 AND corr_index.Year = cy.Year
INNER JOIN agriculture_db.share_of_gdp sog ON sog.Code = cc.ISO3 AND sog.Year = cy.Year
INNER JOIN agriculture_db.urban_population up ON up.Code = cc.ISO3 AND up.Year = cy.Year
INNER JOIN agriculture_db.fertilizer_use fu ON fu.ISO3 = cc.ISO3 AND fu.Year = cy.Year
INNER JOIN agriculture_db.people_employed_in_agriculture peia ON peia.ISO3 = cc.ISO3 AND peia.Year = cy.Year
INNER JOIN agriculture_db.freshwater_withdrawal fw ON fw.ISO3 = cc.ISO3 AND fw.Year = cy.Year
WHERE cy.Year >= 2000"""
features_df = spark.sql(sql_query)
crop_categories_df = spark.table("agriculture_db.crop_categories")

In [0]:
features_df = features_df.dropna(subset=["Yield", "CorruptionIndex", "FertilizerUse", "FreshwaterWithdrawal"])
features_df = (features_df
    .withColumn("Temperature", F.round(F.col("Temperature"), 2))
)
features_df_category = features_df.join(crop_categories_df, on=["ItemCode", "Item"], how="left")
crops_df = features_df_category.filter(((F.col("Category") == "Vegetables Primary") | 
                                        (F.col("Category") == "Crops, primary") |
                                        (F.col("Category") == "Oilcrops, Oil Equivalent") |
                                        (F.col("Category") == "Citrus Fruit, Total") |
                                        (F.col("Category") == "Cereals, primary") |
                                        (F.col("Category") == "Oilcrops, Cake Equivalent") |
                                        (F.col("Category") == "Sugar Crops Primary") |
                                        (F.col("Category") == "Fruit Primary") |
                                        (F.col("Category") == "Oilcrops Primary") |
                                        (F.col("Category") == "Fibre Crops Primary") |
                                        (F.col("Category") == "Fibre Crops, Fibre Equivalent") |
                                        (F.col("Category") == "Roots and Tubers, Total"))
                                       & (F.col("Element") == "Yield")
                                       & (F.col("Yield") > 0))

In [0]:
crops = crops_df.groupBy("Item").agg(F.countDistinct("Country").alias("CountryCount")) \
    .where((F.col("CountryCount") > 90) & ~(F.col("Item").rlike("Other|Primary")))
crop_countries = crops_df.groupBy("Country").agg(F.countDistinct("Item").alias("ItemCount")) \
    .where(F.col("ItemCount") > 80)
crops_filtered_df = crops_df.join(crops.select("Item"), "Item", how="inner").join(crop_countries.select("Country"), "Country", how="inner")

In [0]:
country_data = crops_filtered_df.dropDuplicates(["Year", "Country"]).toPandas()
country_data["TotalPopulation"] = country_data["UrbanPopulation"] + country_data["RuralPopulation"]

In [0]:
fig = sns.lineplot(data=country_data, x="Year", y="Temperature", hue="Country", palette="tab20", linewidth=2)
sns.move_legend(fig, "upper left", bbox_to_anchor=(1, 1))

plt.show()

In [0]:
temperature_intervals = [(5, 10), (10, 13), (13, 16), (16, 19), (19, 22), (22, 25)]

for lower, upper in temperature_intervals:
    temperatures = country_data.groupby("Country")["Temperature"].agg(["min", "max"]).reset_index()

    valid_countries = temperatures[
        (temperatures["min"] >= lower) & (temperatures["max"] <= upper)
    ]["Country"]

    filtered_data = country_data[country_data["Country"].isin(valid_countries)]

    if not filtered_data.empty:
        fig = px.line(
            filtered_data,
            x="Year",
            y="Temperature",
            color="Country",
            title=f"Countries with Temperatures between {lower}°C and {upper}°C",
            labels={"Temperature": "Temperature", "Year": "Year"},
            height=650,
            symbol="Country",
        )

        fig.update_layout(
            legend=dict(
                title="Country",
                x=1,
                y=1,
                traceorder="normal",
                orientation="v",
                xanchor="left",
                yanchor="top",
            ),
            xaxis_title="Year",
            yaxis_title="Temperature",
        )

        fig.show()

In [0]:
fig = px.scatter(
    country_data, 
    x="UrbanPopulation", 
    y="RuralPopulation", 
    size="TotalPopulation", 
    color="Country", 
    hover_name="Country", 
    title="Urban vs. Rural Population by Country",
    labels={"UrbanPopulation": "Urban Population", "RuralPopulation": "Rural Population"},
    template="plotly_dark"
)

fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color="black")))
fig.update_layout(clickmode="event+select")

fig.show()

In [0]:
high_population_filter = ["China, mainland", "India"]
country_data_high_population = country_data[country_data["Country"].isin(high_population_filter)]
lower_population_filter = country_data.groupby("Country")["TotalPopulation"].min() <= 10e6
lower_population_filter = lower_population_filter[lower_population_filter].index.tolist()
country_data_lower_population = country_data[country_data["Country"].isin(lower_population_filter)]
country_population = country_data[~country_data["Country"].isin(high_population_filter + lower_population_filter)]

In [0]:
fig = px.scatter(
    country_data_high_population,
    x="UrbanPopulation",
    y="RuralPopulation",
    size="TotalPopulation",
    color="Country",
    hover_name="Country",
    animation_frame="Year",
    title="Urban vs. Rural Population",
    labels={"UrbanPopulation": "Urban Population", "RuralPopulation": "Rural Population"},
    template="plotly_dark"
)

y_min = country_data_high_population["RuralPopulation"].min() * 0.9
y_max = country_data_high_population["RuralPopulation"].max() * 1.1
x_min = country_data_high_population["UrbanPopulation"].min() * 0.9
x_max = country_data_high_population["UrbanPopulation"].max() * 1.1
fig.update_layout(
    xaxis=dict(range=[x_min, x_max]),
    yaxis=dict(range=[y_min, y_max]),
    clickmode="event+select"
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color="black")))

fig.show()

In [0]:
population_intervals = [(0, 10e6), (10e6, 50e6), (50e6, 100e6)]

for lower, upper in population_intervals:
    populations = country_data.groupby("Country")["TotalPopulation"].agg(["min"]).reset_index()
    
    valid_countries = populations[
        (populations["min"] >= lower) & (populations["min"] <= upper)
    ]["Country"]
    
    filtered_data = country_data[country_data["Country"].isin(valid_countries)]

    if not filtered_data.empty:
        fig = px.scatter(
            filtered_data, 
            x="UrbanPopulation", 
            y="RuralPopulation", 
            size="TotalPopulation", 
            color="Country", 
            hover_name="Country", 
            title=f"Urban vs. Rural Population (Population: {int(lower/1e6)}M - {int(upper/1e6)}M)",
            labels={"UrbanPopulation": "Urban Population", "RuralPopulation": "Rural Population"},
            template="plotly_dark"
        )

        fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color="black")))
        fig.update_layout(clickmode="event+select")

        fig.show()

In [0]:
fig = px.scatter(
    country_population,
    x="UrbanPopulation",
    y="RuralPopulation",
    size="TotalPopulation",
    color="Country",
    animation_frame="Year",
    hover_name="Country",
    title="Population Over Time",
    size_max=60
)

y_min = country_population["RuralPopulation"].min() * -10.0
y_max = country_population["RuralPopulation"].max() * 1.1
x_min = country_population["UrbanPopulation"].min() * -3.0
x_max = country_population["UrbanPopulation"].max() * 1.1
fig.update_layout(
    xaxis=dict(range=[x_min, x_max]),
    yaxis=dict(range=[y_min, y_max]),
    clickmode="event+select"
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color="black")))

In [0]:
cpi_intervals = [(10, 35), (34, 55), (55, 90)]

for lower, upper in cpi_intervals:
    corruption_indexes = country_data.groupby("Country")["CorruptionIndex"].agg(["min", "max"]).reset_index()

    valid_countries = corruption_indexes[
        (corruption_indexes["min"] >= lower) & (corruption_indexes["max"] <= upper)
    ]["Country"]

    filtered_data = country_data[country_data["Country"].isin(valid_countries)]

    if not filtered_data.empty:
        fig = px.line(
            filtered_data,
            x="Year",
            y="CorruptionIndex",
            color="Country",
            title=f"Countries with Corruption Indexes between {lower} and {upper}",
            labels={"CorruptionIndex": "Corruption Index", "Year": "Year"},
            height=650,
            symbol="Country",
        )

        fig.update_layout(
            legend=dict(
                title="Country",
                x=1,
                y=1,
                traceorder="normal",
                orientation="v",
                xanchor="left",
                yanchor="top",
            ),
            xaxis_title="Year",
            yaxis_title="Corruption Index",
        )

        fig.show()

In [0]:
fig = px.scatter(
    country_data,
    x="CorruptionIndex",
    y="GDP",
    size="CorruptionIndex",
    color="Country",
    animation_frame="Year",
    hover_name="Country",
    title="GDP vs CPI"
)

y_min = country_data["GDP"].min() * -5.0
y_max = country_data["GDP"].max() * 1.5
x_min = country_data["CorruptionIndex"].min() * 1.5
x_max = country_data["CorruptionIndex"].max() * 1.5
fig.update_layout(
    xaxis=dict(range=[x_min, x_max]),
    yaxis=dict(range=[y_min, y_max]),
    clickmode="event+select"
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=1, color="black")))

In [0]:
years = sorted(country_data["Year"].unique())

traces = []
for i, year in enumerate(years):
    data_year = country_data[country_data["Year"] == year]
    trace = go.Bar(
        x=data_year["FertilizerUse"],
        y=data_year["Country"],
        orientation="h",
        name=str(year),
        visible=(i == 0)
    )
    traces.append(trace)

buttons = []
for i, year in enumerate(years):
    visibility = [False] * len(years)
    visibility[i] = True
    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[{"visible": visibility},
                  {"title": f"Fertilizer Use by Country – {year}"}]
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Fertilizer Use by Country – {years[0]}",
    xaxis_title="Fertilizer Use (Tonnes)",
    yaxis_title="Country",
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.5,
        xanchor="center",
        y=1.15,
        yanchor="top"
    )]
)

fig.show()

In [0]:
years = sorted(country_data["Year"].unique())

traces = []
for i, year in enumerate(years):
    data_year = country_data[country_data["Year"] == year]
    trace = go.Choropleth(
        locations=data_year["ISO3"],
        z=data_year["PeopleEmployedInAgriculture"],
        locationmode="ISO-3",
        colorscale="Turbo",
        colorbar_title="People",
        text=data_year["Country"],
        name=str(year),
        visible=(i == 0)
    )
    traces.append(trace)

buttons = []
for i, year in enumerate(years):
    visibility = [False] * len(years)
    visibility[i] = True
    buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[{"visible": visibility},
                  {"title": f"People Employed in Agriculture – {year}"}]
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"People Employed in Agriculture – {years[0]}",
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="equirectangular"
    ),
    width=1000,
    height=600,
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.5,
        xanchor="center",
        y=1.1,
        yanchor="top"
    )]
)

fig.show()
